# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

## Solution: combine first, then clean

We will use the first approach suggested in the lab:

1. Read all three CSV files.
2. Standardize their column names before concatenating them. This prevents equivalent columns such as `ST` and `state` from becoming separate columns.
3. Stack the rows with `pd.concat(..., axis=0)`.
4. Clean values, data types, missing values, and duplicates once on the combined table.

This approach keeps one consistent cleaning pipeline and is convenient when all files represent the same kind of record.


In [ ]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
file_urls = [base_url + f"file{i}.csv" for i in range(1, 4)]

dataframes = [pd.read_csv(url) for url in file_urls]

for number, frame in enumerate(dataframes, start=1):
    print(f"file{number}.csv shape: {frame.shape}")
    print(frame.columns.tolist())
    print()


### Step 1 — standardize column names

The helper function converts names to lowercase `snake_case` and replaces `st` with the clearer name `state`. It returns a copy, so the raw DataFrames are not modified accidentally.


In [ ]:
def clean_column_names(frame):
    cleaned = frame.copy()
    cleaned.columns = (
        cleaned.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return cleaned.rename(columns={"st": "state"})


dataframes = [clean_column_names(frame) for frame in dataframes]

# axis=0 means stack rows. ignore_index=True creates one continuous index.
combined_df = pd.concat(dataframes, axis=0, ignore_index=True, sort=False)

print("Combined shape before cleaning:", combined_df.shape)
display(combined_df.head())


### Step 2 — standardize invalid values and formats

We map spelling variants and abbreviations to common labels. Customer lifetime value is cleaned by removing `%` and converting it to a number.

The complaint column can contain encoded text such as `1/5/00`, where the middle component is the actual complaint count. The code also accepts files where this column is already numeric.


In [ ]:
combined_df["gender"] = combined_df["gender"].replace({
    "Femal": "F", "female": "F", "Female": "F",
    "Male": "M", "male": "M"
})

combined_df["state"] = combined_df["state"].replace({
    "AZ": "Arizona", "Cali": "California", "WA": "Washington"
})

combined_df["education"] = combined_df["education"].replace({
    "Bachelors": "Bachelor"
})

combined_df["vehicle_class"] = combined_df["vehicle_class"].replace({
    "Sports Car": "Luxury",
    "Luxury SUV": "Luxury",
    "Luxury Car": "Luxury"
})

combined_df["customer_lifetime_value"] = pd.to_numeric(
    combined_df["customer_lifetime_value"]
    .astype("string")
    .str.replace("%", "", regex=False),
    errors="coerce"
)

complaints_as_text = combined_df["number_of_open_complaints"].astype("string")
has_slashes = complaints_as_text.str.contains("/", na=False)
complaint_value = complaints_as_text.where(
    ~has_slashes,
    complaints_as_text.str.split("/").str[1]
)
combined_df["number_of_open_complaints"] = pd.to_numeric(
    complaint_value, errors="coerce"
)

numeric_columns = [
    "income", "monthly_premium_auto", "total_claim_amount"
]
combined_df[numeric_columns] = combined_df[numeric_columns].apply(
    pd.to_numeric, errors="coerce"
)

display(combined_df.dtypes)


### Step 3 — handle nulls and duplicates

- Completely blank rows and rows without a customer ID are removed because they cannot represent identifiable customer records.
- Remaining numeric gaps are filled with each column's median, which is resistant to large outliers.
- Remaining categorical gaps are filled with the mode, the most frequent valid value.
- Exact duplicate rows are removed, keeping the first copy.

The checks at the end verify the cleaning results. We then save the combined clean dataset for reuse.


In [ ]:
combined_df = combined_df.dropna(how="all")
combined_df = combined_df.dropna(subset=["customer"]).copy()

numeric_columns = combined_df.select_dtypes(include="number").columns
categorical_columns = combined_df.select_dtypes(exclude="number").columns

for column in numeric_columns:
    combined_df[column] = combined_df[column].fillna(
        combined_df[column].median()
    )

for column in categorical_columns:
    if combined_df[column].isna().any():
        combined_df[column] = combined_df[column].fillna(
            combined_df[column].mode().iloc[0]
        )

duplicates_before = combined_df.duplicated().sum()
combined_df = (
    combined_df
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

print("Exact duplicates removed:", duplicates_before)
print("Remaining null values:", combined_df.isna().sum().sum())
print("Final combined shape:", combined_df.shape)

combined_df.to_csv("combined_customer_data_clean.csv", index=False)
display(combined_df.head())


**Interpretation:** `concat` increases the number of rows rather than adding unrelated columns. Standardizing names before concatenation is essential: pandas aligns columns by their labels, so inconsistent labels would otherwise create duplicate logical columns filled with nulls.


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

## Solution setup

Load the marketing dataset and inspect its structure. We standardize column names defensively so the remaining code consistently uses lowercase `snake_case`. Columns beginning with `unnamed` usually come from a previously saved DataFrame index, so they are removed because they are not business variables.


In [ ]:
marketing_url = (
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
    "marketing_customer_analysis_clean.csv"
)
marketing_df = pd.read_csv(marketing_url)

marketing_df.columns = (
    marketing_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

index_columns = [
    column for column in marketing_df.columns
    if column.startswith("unnamed")
]
marketing_df = marketing_df.drop(columns=index_columns)

print("Shape:", marketing_df.shape)
display(marketing_df.head())
display(marketing_df.dtypes)


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

### 1. Total revenue by sales channel

In this dataset, **customer lifetime value** is the available revenue/value measure, so we use it as the requested revenue metric. `pivot_table()` groups rows by `sales_channel` and sums customer lifetime value in each group. `round(2)` formats the result to two decimal places.

The table is sorted from highest to lowest so the leading channel appears first.


In [ ]:
revenue_by_channel = (
    marketing_df.pivot_table(
        index="sales_channel",
        values="customer_lifetime_value",
        aggfunc="sum"
    )
    .rename(columns={"customer_lifetime_value": "total_revenue"})
    .round(2)
    .sort_values("total_revenue", ascending=False)
)

display(revenue_by_channel)

top_channel = revenue_by_channel["total_revenue"].idxmax()
top_revenue = revenue_by_channel.loc[top_channel, "total_revenue"]
revenue_share = top_revenue / revenue_by_channel["total_revenue"].sum() * 100

print(f"Highest-revenue channel: {top_channel}")
print(f"Revenue: {top_revenue:,.2f}")
print(f"Share of revenue: {revenue_share:.2f}%")


**How to interpret it:** The first row is the channel associated with the greatest total customer value. However, a high total can result from either more customers or higher-value customers. A useful next step would compare both customer count and average value per channel before deciding where marketing is most efficient.


### 2. Average customer lifetime value by gender and education

Here, gender forms the rows and education levels form the columns. `aggfunc="mean"` calculates the average customer lifetime value for every gender–education combination. `margins=True` adds overall averages, and rounding improves readability.


In [ ]:
average_clv = marketing_df.pivot_table(
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean",
    margins=True,
    margins_name="All"
).round(2)

display(average_clv)

# Stack only the gender/education cells—not the margin—to locate the maximum.
detail = average_clv.drop(index="All").drop(columns="All")
highest_group = detail.stack().idxmax()
highest_average = detail.stack().max()

print(
    f"Highest average CLV: gender={highest_group[0]}, "
    f"education={highest_group[1]}, value={highest_average:,.2f}"
)


**How to interpret it:** Compare values across a row to see how education relates to average CLV within one gender, and down a column to compare genders at one education level. Small groups can produce unstable averages, so group counts should be checked before drawing strong conclusions.


### Bonus: complaints by policy type and month in long format

First, a pivot table sums the number of open complaints for every policy-type/month combination. That result is a **wide** table because each month is a separate column.

`reset_index().melt(...)` converts it to **long format**, producing one row per policy type and month. Long format is especially convenient for sorting, filtering, and plotting.


In [ ]:
complaints_wide = marketing_df.pivot_table(
    index="policy_type",
    columns="month",
    values="number_of_open_complaints",
    aggfunc="sum",
    fill_value=0
)

complaints_long = (
    complaints_wide
    .reset_index()
    .melt(
        id_vars="policy_type",
        var_name="month",
        value_name="number_of_complaints"
    )
    .sort_values(
        ["number_of_complaints", "policy_type"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(complaints_long)

highest_complaint_row = complaints_long.iloc[0]
print(
    "Highest combination: "
    f"{highest_complaint_row['policy_type']} in month "
    f"{highest_complaint_row['month']} with "
    f"{highest_complaint_row['number_of_complaints']} complaints"
)


**How to interpret it:** Because the long table is sorted in descending order, its first row identifies the policy type and month with the largest total number of open complaints. This is a workload signal, not necessarily a quality rate: policy categories with more customers can naturally generate more complaints. Dividing by the number of policies in each group would provide a complaint rate.
